## Q1. 파일 열고 크기 확인하기

파일을 표로 불러오고, 몇 건이 몇 열로 되어 있는지 확인하세요.

In [6]:
import pandas as pd

df = pd.read_csv('../../data/day01_bottling.csv')

# 1. 전체 행 수와 열 수
print('행 수, 열 수:', df.shape)

행 수, 열 수: (4800, 13)


---
## Q2. 검사 결과 살펴보기

`result` 열에 합격과 재검이 각각 몇 건인지, 그리고 재검이 전체의 몇 %인지 확인하세요.

In [13]:
# 2. result 열의 값 종류별 건수와 비율
print('\nresult 건수:')
print(df['result'].value_counts())
print('\nresult 비율:')
print(df['result'].value_counts(normalize=True).mul(100).round(2))


result 건수:
result
합격    4554
재검     246
Name: count, dtype: int64

result 비율:
result
합격    94.88
재검     5.12
Name: proportion, dtype: float64


---
## Q3. 제품 규격별로 몇 건씩인가

In [15]:
df["product"].value_counts()

product
500mL    2428
1L       1397
2L        975
Name: count, dtype: int64

---
## Q4. 유독 뜨겁게 밀봉한 묶음 찾기

In [16]:
# ascending=False — 큰 값이 위로 오게(내림차순)
df.sort_values("seal_temp", ascending=False).head(5)[
    ["lot_id", "line_id", "seal_temp", "result"]
]

,lot_id,line_id,seal_temp,result
4554,L04555,F-2,209.5,재검
3946,L03947,F-2,207.2,재검
4031,L04032,F-2,207.0,재검
4703,L04704,F-2,205.9,재검
4383,L04384,F-2,205.5,재검


---
## Q5. 라인별로 재검률이 다른가

In [18]:
# result == "재검" 이면 참(1), 아니면 거짓(0)
# 그것의 평균을 내면 곧 재검 비율이 된다
(df.groupby("line_id")["result"].apply(lambda s: (s == "재검").mean()) * 100).round(2)

line_id
F-1    4.65
F-2    5.29
F-3    5.65
Name: result, dtype: float64

---
## Q6. 비어 있는 칸 찾기

In [19]:
# 열마다 비어 있는 칸 개수
print(df.isna().sum())

# 빈칸이 하나라도 있는 열이 몇 개인지
print("빈칸 있는 열 개수:", (df.isna().sum() > 0).sum())

lot_id                0
produced_at           0
plant_code            0
line_id               0
shift                 0
product               0
fill_error            0
cap_torque          139
seal_temp             0
line_speed            0
ambient_temp          0
ambient_humidity     74
result                0
dtype: int64
빈칸 있는 열 개수: 2


---
## Q7. 캡 토크로 거르면 몇 건이 빠질까

In [20]:
print("전체:", len(df))
print("2.0 초과:", len(df[df["cap_torque"] > 2.0]))
print("2.0 이하:", len(df[df["cap_torque"] <= 2.0]))
print("합:", len(df[df["cap_torque"] > 2.0]) + len(df[df["cap_torque"] <= 2.0]))

전체: 4800
2.0 초과: 3340
2.0 이하: 1321
합: 4661


* 결측치는 세어지지 않음.<br>Q6에서 기록되지 않은 캡 토크 칸은 139개<br>지금 모자란 값도 139개로 일치

---
## Q8. 교대조에 따라 다른가

In [21]:
df.groupby("shift")["cap_torque"].mean().round(3)

shift
야간    2.040
오후    2.084
주간    2.090
Name: cap_torque, dtype: float64

---
## Q9. 밀봉 온도에 관리선 긋기 ⭐ (이 미션의 핵심)

In [22]:
평균 = df["seal_temp"].mean()
표준편차 = df["seal_temp"].std()

위선 = 평균 + 3 * 표준편차
아래선 = 평균 - 3 * 표준편차

print("평균:", round(평균, 2))
print("관리 상한:", round(위선, 2))
print("관리 하한:", round(아래선, 2))

# 위로 넘었거나 아래로 넘은 묶음만 고른다
벗어남 = df[(df["seal_temp"] > 위선) | (df["seal_temp"] < 아래선)]

print()
print("관리선 벗어난 건수:", len(벗어남))
print(벗어남["result"].value_counts())
print()
print("전체 재검:", (df["result"] == "재검").sum())

평균: 180.35
관리 상한: 188.3
관리 하한: 172.39

관리선 벗어난 건수: 52
result
합격    35
재검    17
Name: count, dtype: int64

전체 재검: 246


---
## Q10. 열마다 한 줄로 요약하기

In [23]:
숫자열 = ["fill_error", "cap_torque", "seal_temp",
          "line_speed", "ambient_temp", "ambient_humidity"]

진단표 = pd.DataFrame({
    "빈칸비율(%)": (df[숫자열].isna().sum() / len(df) * 100).round(2),
    "값종류수": df[숫자열].nunique(),
    "표준편차": df[숫자열].std().round(3),
    "최솟값": df[숫자열].min(),
    "최댓값": df[숫자열].max(),
})

진단표

,빈칸비율(%),값종류수,표준편차,최솟값,최댓값
fill_error,0.00,570,1.093,-5.49,5.50
cap_torque,2.90,124,0.165,1.21,2.58
seal_temp,0.00,183,2.652,173.40,209.50
line_speed,0.00,81,12.048,351.00,442.00
ambient_temp,0.00,193,3.235,14.30,36.30
ambient_humidity,1.54,423,7.918,22.40,82.50
